# Factual Recall γ — Phase 1-4 (Llama-3.1-70B-Instruct-4bit, NF4, hihp/hilp schema)

Single-model run for the 70B scale-arm. Mirrors `factual_recall_standard.ipynb` cell structure; differences vs standard:

- **Cell 1**: GDrive mount FIRST, then VRAM ≥ 40 GB assertion (70B-NF4 needs ~37 GB; T4/L4 will raise).
- **Cell 2**: `MODEL_SPECS = [(cmarkea/Meta-Llama-3.1-70B-Instruct-4bit, llama-3.1-70b-instruct-4bit)]`, batch ladders, `_sweep_memory()` cadence.
- **Cell 4**: `BitsAndBytesConfig` NF4 load + Pattern-A batched per-head patching (`patched_logits_batched_heads`) + batched group patching (`patched_logits_batched_trials`).
- **Cell 5**: Phase 2 uses `HEAD_BATCH_SIZE` (default 32, ladder [32,16,8,4]) so 5120 individual-head forwards run as batched traces; Phase 4 uses `TRIAL_BATCH_SIZE` (default 8) to batch 80 trials per (ordering, ratio) trace.
- Per-trial CSV save on Phase 2; per-(ordering,ratio) save on Phase 4 (lessons from Prospect 70B kernel-stall).

**Cell labels**: hihp (hi-imp hi-pert), hilp (hi-imp lo-pert), C (lo-imp hi-pert), D (lo-imp lo-pert)
**Orderings (7)**: hihp_imp_desc, hihp_imp_asc, hilp_imp_desc, hilp_imp_asc, C_imp_asc, D_imp_asc, diag_rank_asc

**Output paths** (unchanged from existing FR convention):
`/content/drive/MyDrive/WCC/factual_recall/llama-3.1-70b-instruct-4bit/{10_collection,20_scoring,30_patching}/...`


In [ ]:
# ── Cell 1: Drive mount FIRST → install → HF login → VRAM check ──
from google.colab import drive, runtime
drive.mount('/content/drive')

!pip install -q -U "transformers" "accelerate" "bitsandbytes>=0.46.1" "nnsight" "scipy" "pandas" "scikit-learn"

import os
os.environ['HF_TOKEN'] = '<YOUR_HF_TOKEN>'
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
from huggingface_hub import login as _hf_login
_hf_login(token=os.environ['HF_TOKEN'])
print('HF_TOKEN set + huggingface_hub.login() called.')

import json, gc, time, math, random, traceback
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import torch

def log(msg):
    print(f'[{datetime.now().strftime("%H:%M:%S")}] {msg}', flush=True)

log(f'Torch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'GPU runtime required for 70B-NF4'
_props = torch.cuda.get_device_properties(0)
_vram_gb = _props.total_memory / 1024**3
log(f'GPU: {_props.name}  VRAM={_vram_gb:.1f} GB')
# 70B 4-bit footprint ~37 GB (Prospect-measured peak). Need ≥40 GB to be safe.
assert _vram_gb >= 39.0, (
    f'70B-NF4 needs ≥40 GB VRAM (Prospect-measured peak ≈ 37 GB). '
    f'Current GPU has {_vram_gb:.1f} GB. Switch to a Colab G4 (95.6GB) or higher-VRAM runtime.'
)


In [ ]:
# ── Cell 2: Config (70B single-model + batch ladders, Colab G4 95.6GB tuned) ──
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MODEL_SPECS = [
    ('cmarkea/Meta-Llama-3.1-70B-Instruct-4bit', 'llama-3.1-70b-instruct-4bit'),
]
MODEL_DTYPE = 'nf4'
PATCH_MECHANISM = 'input_overwrite_v5_1_pattern'
PATCH_MECHANISM_VERSION = 'v2_canonical_patch_mechanism'

# ===== Batch sizes — tuned for Colab G4 GPU 95.6GB (Prospect-measured 70B-NF4 weight footprint ~37 GB) =====
# Target: keep peak GPU < 80 GB to leave 16 GB safety margin.
# Phase 2 per-head: B prompts replicated, each with single-head patch
HEAD_BATCH_SIZE = 64
HEAD_BATCH_LADDER = [64, 48, 32, 16, 8, 4]

# Phase 4 group: B trials per trace, each gets the SAME group patch but DIFFERENT source vec
TRIAL_BATCH_SIZE = 16
TRIAL_BATCH_LADDER = [16, 12, 8, 4, 2, 1]

# Memory hygiene
SWEEP_EVERY_N_TRACES = 4

# Standard FR config — UNCHANGED from factual_recall_standard.ipynb
RATIOS = [0.05, 0.10, 0.20, 0.30, 0.50]
CELLS  = ['hihp', 'hilp', 'C', 'D', 'diag']
ORDERINGS = [
    'hihp_imp_desc', 'hihp_imp_asc',
    'hilp_imp_desc', 'hilp_imp_asc',
    'C_imp_asc', 'D_imp_asc',
    'diag_rank_asc',
]
REQUIRED_ORDERINGS = set(ORDERINGS)
N_TRIALS_TARGET = 80

DRIVE_BASE = '/content/drive/MyDrive/WCC'
DATA_DIR   = f'{DRIVE_BASE}/factual_recall/00_data'
OUT_BASE   = f'{DRIVE_BASE}/factual_recall'

assert os.path.exists(f'{DATA_DIR}/trial_definitions.csv'), \
    f'Phase 0 output missing — run factual_recall_data_prep.ipynb first'

all_trials = pd.read_csv(f'{DATA_DIR}/trial_definitions.csv')
log(f'Phase 0 trials: {len(all_trials)} across {all_trials["relation"].nunique()} relations')

n_per_rel = max(1, N_TRIALS_TARGET // all_trials['relation'].nunique())
# Use groupby().head() instead of groupby().apply(): pandas 2.2+ drops grouping column
# inside apply(group_keys=False) results. .head() preserves all columns including 'relation'.
sampled = (all_trials
    .sort_values(['relation', 'trial_id'], kind='mergesort')
    .groupby('relation', group_keys=False, sort=False, as_index=False)
    .head(n_per_rel)
    .reset_index(drop=True))
if len(sampled) > N_TRIALS_TARGET:
    sampled = sampled.sample(N_TRIALS_TARGET, random_state=SEED).reset_index(drop=True)
trials_df = sampled.copy()
assert 'relation' in trials_df.columns, f'Invariant violation: relation column dropped — got {list(trials_df.columns)}'
log(f'Sampled trials: {len(trials_df)} across {trials_df["relation"].nunique()} relations')
log(f'Batch config (Colab G4 95.6GB): HEAD_BATCH={HEAD_BATCH_SIZE} (ladder={HEAD_BATCH_LADDER}) | TRIAL_BATCH={TRIAL_BATCH_SIZE} (ladder={TRIAL_BATCH_LADDER})')
log(f'  Phase 2 forecast traces: {math.ceil(80 * 5120 / HEAD_BATCH_SIZE)} = ~{int(80 * 5120 / HEAD_BATCH_SIZE * 6 / 3600)} h at 6 s/trace')
log(f'  Phase 4 forecast traces: {math.ceil(7 * 5 * 80 / TRIAL_BATCH_SIZE)} = ~{int(7 * 5 * 80 / TRIAL_BATCH_SIZE * 12 / 60)} min at 12 s/trace')


In [ ]:
# ── Cell 3: Common helpers (+ memory diagnostics) ──
import nnsight as nns
from nnsight import LanguageModel
log(f'nnsight version: {nns.__version__}')

import psutil as _psutil_mem  # for host RAM diagnostics

def _mem_status():
    vm = _psutil_mem.virtual_memory()
    host_gb = vm.used / 1024**3
    host_total = vm.total / 1024**3
    gpu_gb = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
    gpu_peak = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
    return f'host={host_gb:.1f}/{host_total:.1f}GB gpu={gpu_gb:.1f}GB peak={gpu_peak:.1f}GB'

def _sweep_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

def get_token_id_with_space(tokenizer, text):
    prefix = 'The answer is'
    full = tokenizer.encode(prefix + ' ' + str(text), add_special_tokens=False)
    pref = tokenizer.encode(prefix, add_special_tokens=False)
    return full[len(pref)]

def unload_model(model):
    try: del model
    except Exception: pass
    _sweep_memory()

def ensure_dir(p):
    os.makedirs(p, exist_ok=True); return p

def phase_dir(model_short, phase):
    return ensure_dir(f'{OUT_BASE}/{model_short}/{phase}')

def phase_done(model_short, phase):
    return os.path.exists(f'{phase_dir(model_short, phase)}/config.json')

def mark_phase_done(model_short, phase, meta, start_time=None):
    payload = {**meta, 'completed_at': datetime.now().isoformat()}
    if start_time is not None:
        payload['elapsed_sec'] = round(time.time() - start_time, 2)
    with open(f'{phase_dir(model_short, phase)}/config.json', 'w') as f:
        json.dump(payload, f, indent=2)

def phase4_all_orderings_complete(model_short):
    csv_path = f'{phase_dir(model_short, "30_patching")}/behavioral_gamma.csv'
    if not os.path.exists(csv_path): return False
    try: df = pd.read_csv(csv_path)
    except Exception: return False
    if 'ordering' not in df.columns: return False
    return REQUIRED_ORDERINGS.issubset(set(df['ordering'].unique()))

def update_phase4_config(model_short, meta):
    cfg_path = f'{phase_dir(model_short, "30_patching")}/config.json'
    cfg = {}
    if os.path.exists(cfg_path):
        try: cfg = json.load(open(cfg_path))
        except Exception: cfg = {}
    prev = cfg.get('completed_at')
    if prev and 'first_completed_at_asc_only' not in cfg:
        cfg['first_completed_at_asc_only'] = prev
    cfg.update(meta)
    cfg['last_updated_at'] = datetime.now().isoformat()
    cfg.pop('completed_at', None)
    with open(cfg_path, 'w') as f:
        json.dump(cfg, f, indent=2)


In [ ]:
# ── Cell 4: Load + clean/collect/patch (70B NF4, batched Pattern A) ──
from transformers import BitsAndBytesConfig

def load_model(model_id):
    log(f'Loading {model_id}  (NF4 4-bit pre-quantized — uses model\'s embedded quant_config)')
    # cmarkea/Meta-Llama-3.1-70B-Instruct-4bit is pre-quantized; do NOT pass a new quant_config.
    model = LanguageModel(model_id, device_map='auto')
    cfg = model.config
    arch = {
        'num_layers' : cfg.num_hidden_layers,
        'num_heads'  : cfg.num_attention_heads,
        'head_dim'   : getattr(cfg, 'head_dim', cfg.hidden_size // cfg.num_attention_heads),
        'hidden_size': cfg.hidden_size,
        'vocab_size' : cfg.vocab_size,
    }
    arch['total_heads'] = arch['num_layers'] * arch['num_heads']
    log(f'  arch: L={arch["num_layers"]} H={arch["num_heads"]} D={arch["head_dim"]} total={arch["total_heads"]}')
    log(f'  post-load memory: {_mem_status()}')
    return model, arch


@torch.no_grad()
def clean_logits_last(model, prompt):
    with model.trace(prompt) as tracer:
        logits = model.output.logits[0, -1, :].save()
    return logits.detach().float().cpu().numpy()


@torch.no_grad()
def collect_attn_input_per_layer(model, arch, prompt):
    """Capture o_proj input (= attention output before W_O) for each layer at last token."""
    L, H, D = arch['num_layers'], arch['num_heads'], arch['head_dim']
    saves = {}
    with model.trace(prompt) as tracer:
        for i in range(L):
            saves[i] = model.model.layers[i].self_attn.o_proj.input[0, -1, :].save()
    if len(saves) != L:
        raise RuntimeError(f'Trace failed: captured {len(saves)}/{L} layers')
    return np.stack([saves[i].detach().float().cpu().numpy().reshape(H, D) for i in range(L)])


@torch.no_grad()
def patched_logits_single_head(model, arch, prompt, source_vec, layer_idx, head_idx):
    """Single-head patch (used as fallback when batched call OOMs at B=1)."""
    H, D = arch['num_heads'], arch['head_dim']
    src = torch.as_tensor(source_vec, dtype=torch.bfloat16, device='cuda')
    with model.trace(prompt) as tracer:
        proj = model.model.layers[int(layer_idx)].self_attn.o_proj
        col_s = int(head_idx) * D
        proj.input[0, -1, col_s:col_s+D] = src[int(layer_idx), int(head_idx)]
        logits = model.output.logits[0, -1, :].save()
    return logits.detach().float().cpu().numpy()


@torch.no_grad()
def patched_logits_batched_heads(model, arch, prompt, source_vec, head_list, target_id):
    """Batched per-head patching via prompt-replication.
    Trace runs B prompts simultaneously; each batch index gets a *different* single-head patch.
    Returns: np.array shape (B,) of patched target_logits.
    """
    B = len(head_list)
    if B == 0:
        return np.empty(0, dtype=np.float32)
    H, D = arch['num_heads'], arch['head_dim']
    src = torch.as_tensor(source_vec, dtype=torch.bfloat16, device='cuda')
    prompts = [prompt] * B
    with model.trace(prompts) as tracer:
        for bi, (l, h) in enumerate(head_list):
            proj = model.model.layers[int(l)].self_attn.o_proj
            col_s = int(h) * D
            proj.input[bi, -1, col_s:col_s+D] = src[int(l), int(h)]
        tgt = model.output.logits[:, -1, int(target_id)].save()
    return tgt.detach().float().cpu().numpy()


@torch.no_grad()
def patched_logits_batched_trials(model, arch, prompts, source_vecs, patch_heads):
    """Batched group patching across B trials.
    Each trial gets the SAME group of patch_heads but a DIFFERENT source vec.
    Returns: np.array shape (B, vocab_size) of full patched logits.
    """
    B = len(prompts)
    if B == 0:
        return np.empty((0, arch['vocab_size']), dtype=np.float32)
    H, D = arch['num_heads'], arch['head_dim']
    src_stack = torch.as_tensor(np.stack(source_vecs), dtype=torch.bfloat16, device='cuda')
    sorted_patches = sorted(patch_heads, key=lambda lh: lh[0])
    with model.trace(prompts) as tracer:
        for layer_idx, head_idx in sorted_patches:
            proj = model.model.layers[int(layer_idx)].self_attn.o_proj
            col_s = int(head_idx) * D
            proj.input[:, -1, col_s:col_s+D] = src_stack[:, int(layer_idx), int(head_idx)]
        logits = model.output.logits[:, -1, :].save()
    return logits.detach().float().cpu().numpy()


def _call_with_batch_ladder(fn, batch_attr_name, ladder_attr_name, *fn_args, **fn_kwargs):
    """Call fn(batch_size, *fn_args) with OOM-fallback through a batch-size ladder.
    fn must accept batch_size as the FIRST argument and return None (saving inside).
    Returns the batch_size that succeeded.
    """
    g = globals()
    ladder = list(g.get(ladder_attr_name))
    current = g.get(batch_attr_name)
    if current not in ladder:
        ladder = [current] + [b for b in ladder if b < current]
    ladder_ptr = ladder.index(current)
    while True:
        b = ladder[ladder_ptr]
        try:
            fn(b, *fn_args, **fn_kwargs)
            g[batch_attr_name] = b
            return b
        except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:
            msg = str(exc).lower()
            if 'out of memory' not in msg and not isinstance(exc, torch.cuda.OutOfMemoryError):
                raise
            _sweep_memory()
            if ladder_ptr >= len(ladder) - 1:
                raise
            ladder_ptr += 1
            log(f'  OOM at batch={b} → ladder down to {ladder[ladder_ptr]}  ({_mem_status()})')


In [ ]:
# ── Cell 5: Phase 1-4 implementations (70B-batched) ──
FAIL_LOG_CHARS = 500
FAIL_LIMIT_PER_TRIAL = 20

# ---------- Phase 1: Clean pass (UNCHANGED — sequential, fast on 80 trials) ----------
def run_clean_pass(model, arch, tokenizer, trials_df, model_short):
    out_path = f'{phase_dir(model_short, "10_collection")}/clean_logits.csv'
    rows = []; t0 = time.time()
    for i, trial in trials_df.iterrows():
        target_id     = get_token_id_with_space(tokenizer, trial['target'])
        competitor_id = get_token_id_with_space(tokenizer, trial['competitor'])
        try:
            lg = clean_logits_last(model, trial['prompt'])
            argmax_id = int(lg.argmax())
            sorted_idx = np.argsort(-lg)
            target_rank = int(np.where(sorted_idx == target_id)[0][0]) + 1
            target_logit = float(lg[target_id])
            competitor_logit = float(lg[competitor_id])
        except Exception as e:
            log(f'  Phase1 WARN trial={trial["trial_id"]}: {type(e).__name__}: {str(e)[:FAIL_LOG_CHARS]}')
            argmax_id = -1; target_rank = -1
            target_logit = float('nan'); competitor_logit = float('nan')
        rows.append({
            'trial_id': trial['trial_id'], 'relation': trial['relation'],
            'subject': trial['subject'], 'target': trial['target'],
            'competitor': trial['competitor'],
            'target_id': target_id, 'competitor_id': competitor_id,
            'argmax_id': argmax_id,
            'argmax_token': tokenizer.decode([argmax_id]) if argmax_id >= 0 else '',
            'target_rank': target_rank, 'target_logit': target_logit,
            'competitor_logit': competitor_logit,
            'margin': target_logit - competitor_logit,
            'is_top1': (target_rank == 1),
            'is_top5': (1 <= target_rank <= 5),
        })
        if (i+1) % 10 == 0:
            log(f'  Phase1 {i+1}/{len(trials_df)}  elapsed={time.time()-t0:.0f}s  {_mem_status()}')
    df = pd.DataFrame(rows); df.to_csv(out_path, index=False)
    log(f'  Phase1 complete  top1={int(df["is_top1"].sum())}  top5={int(df["is_top5"].sum())}  '
        f'elapsed={time.time()-t0:.0f}s')
    return df


# ---------- Phase 2: Per-head importance + perturbation (HEAD_BATCH-ed for 70B) ----------
def run_phase2_scoring(model, arch, tokenizer, trials_df, clean_df, model_short):
    out_imp  = f'{phase_dir(model_short, "20_scoring")}/importance_per_trial_head.csv'
    out_pert = f'{phase_dir(model_short, "20_scoring")}/perturbation_per_trial_head.csv'
    L, H = arch['num_layers'], arch['num_heads']
    clean_lookup = clean_df.set_index('trial_id')

    if os.path.exists(out_imp):
        imp_done = pd.read_csv(out_imp); done_imp = set(imp_done['trial_id'].unique())
        log(f'  Phase2 resume imp: {len(done_imp)} trials done')
    else:
        imp_done = pd.DataFrame(); done_imp = set()
    if os.path.exists(out_pert):
        pert_done = pd.read_csv(out_pert); done_pert = set(pert_done['trial_id'].unique())
    else:
        pert_done = pd.DataFrame(); done_pert = set()

    imp_rows  = list(imp_done.to_dict('records'))
    pert_rows = list(pert_done.to_dict('records'))
    t0 = time.time()
    all_heads = [(l, h) for l in range(L) for h in range(H)]
    n_total_traces_per_trial = math.ceil(len(all_heads) / HEAD_BATCH_SIZE)

    for ti, trial in trials_df.iterrows():
        tid = trial['trial_id']
        if tid in done_imp and tid in done_pert:
            continue
        try:
            target_id = int(clean_lookup.loc[tid, 'target_id'])
            clean_target_logit = float(clean_lookup.loc[tid, 'target_logit'])
        except KeyError:
            log(f'  Phase2 skip {tid}: no clean_df entry'); continue

        try:
            clean_cache   = collect_attn_input_per_layer(model, arch, trial['prompt'])
            corrupt_cache = collect_attn_input_per_layer(model, arch, trial['corrupt_prompt'])
        except Exception as e:
            log(f'  Phase2 FAIL collect {tid}: {type(e).__name__}: {str(e)[:FAIL_LOG_CHARS]}')
            continue

        if tid not in done_pert:
            for l in range(L):
                for h in range(H):
                    l2 = float(np.linalg.norm(clean_cache[l, h] - corrupt_cache[l, h]))
                    pert_rows.append({'trial_id': tid, 'layer': l, 'head': h, 'perturbation_l2': l2})
            pd.DataFrame(pert_rows).to_csv(out_pert, index=False)

        if tid not in done_imp:
            tgt_logits_patched = np.full((L, H), np.nan)
            n_fail = 0; abort_trial = False
            t_trial = time.time()
            traces_done = 0
            batch_start = 0
            while batch_start < len(all_heads) and not abort_trial:
                # Use batch ladder to handle OOM gracefully
                def _do_batch(B):
                    nonlocal batch_start, traces_done
                    batch = all_heads[batch_start:batch_start + B]
                    if not batch:
                        return
                    results = patched_logits_batched_heads(
                        model, arch, trial['prompt'], corrupt_cache, batch, target_id)
                    for (lh_idx, (l, h)), v in zip(enumerate(batch), results):
                        tgt_logits_patched[l, h] = float(v)
                try:
                    used_b = _call_with_batch_ladder(_do_batch, 'HEAD_BATCH_SIZE', 'HEAD_BATCH_LADDER')
                    batch_start += used_b
                    traces_done += 1
                    if traces_done % SWEEP_EVERY_N_TRACES == 0:
                        _sweep_memory()
                except Exception as e:
                    if n_fail < 3:
                        log(f'    Phase2 batch FAIL tid={tid} start={batch_start}: {type(e).__name__}: {str(e)[:FAIL_LOG_CHARS]}')
                    n_fail += 1
                    if n_fail >= FAIL_LIMIT_PER_TRIAL:
                        log(f'  Phase2 ABORT trial {tid} after {n_fail} batch failures')
                        abort_trial = True
                    else:
                        # Skip this batch (NaN remains)
                        batch_start += HEAD_BATCH_SIZE
            for l in range(L):
                for h in range(H):
                    patched_logit = tgt_logits_patched[l, h]
                    delta = (patched_logit - clean_target_logit) if not np.isnan(patched_logit) else float('nan')
                    imp_rows.append({
                        'trial_id': tid, 'layer': l, 'head': h,
                        'clean_target_logit': clean_target_logit,
                        'patched_target_logit': patched_logit if not np.isnan(patched_logit) else float('nan'),
                        'delta_target_logit': delta,
                    })
            pd.DataFrame(imp_rows).to_csv(out_imp, index=False)
            trial_dt = time.time() - t_trial
            elapsed = time.time() - t0
            log(f'  Phase2 trial {ti+1}/{len(trials_df)}  tid={tid}  '
                f'trial_dt={trial_dt:.0f}s  cum={elapsed:.0f}s  '
                f'traces={traces_done}/{n_total_traces_per_trial}  '
                f'B={HEAD_BATCH_SIZE}  {_mem_status()}')
            _sweep_memory()

    imp_df  = pd.DataFrame(imp_rows)
    pert_df = pd.DataFrame(pert_rows)

    imp_head = imp_df.groupby(['layer','head']).agg(
        mean_abs_delta=('delta_target_logit', lambda x: float(np.nanmean(np.abs(x)))),
        mean_signed_delta=('delta_target_logit', lambda x: float(np.nanmean(x))),
        n_trials=('trial_id', 'nunique'),
    ).reset_index()
    imp_head.to_csv(f'{phase_dir(model_short, "20_scoring")}/importance_per_head.csv', index=False)

    pert_head = pert_df.groupby(['layer','head']).agg(
        mean_perturbation_l2=('perturbation_l2', 'mean'),
        std_perturbation_l2=('perturbation_l2', 'std'),
        n_trials=('trial_id', 'nunique'),
    ).reset_index()
    pert_head.to_csv(f'{phase_dir(model_short, "20_scoring")}/perturbation_per_head.csv', index=False)
    log(f'  Phase2 complete  elapsed={time.time()-t0:.0f}s  {_mem_status()}')
    return imp_head, pert_head


# ---------- Phase 3: Cell classification (UNCHANGED) ----------
def classify_cells(imp_head, pert_head, model_short):
    merged = imp_head.merge(pert_head, on=['layer', 'head'])
    merged['importance']   = merged['mean_abs_delta']
    merged['perturbation'] = merged['mean_perturbation_l2']
    imp_med, pert_med = merged['importance'].median(), merged['perturbation'].median()
    def assign(row):
        hi_i = row['importance'] > imp_med
        hi_p = row['perturbation'] > pert_med
        if hi_i and hi_p:     return 'hihp'
        if hi_i and not hi_p: return 'hilp'
        if not hi_i and hi_p: return 'C'
        return 'D'
    merged['cell'] = merged.apply(assign, axis=1)
    merged[['layer','head','importance','perturbation','cell']].to_csv(
        f'{phase_dir(model_short, "20_scoring")}/cell_classification.csv', index=False)
    for c in CELLS:
        n = (merged['cell']==c).sum()
        log(f'  Cell {c}: {n} heads ({n/len(merged)*100:.1f}%)')
    return merged


# ---------- Phase 4: γ group patching (TRIAL_BATCH-ed for 70B) ----------
def build_orderings(cell_df):
    orderings = {}
    for cell_letter in ('hihp', 'hilp'):
        sub = cell_df[cell_df['cell']==cell_letter]
        heads_desc = list(zip(sub.sort_values('importance', ascending=False)['layer'].astype(int),
                              sub.sort_values('importance', ascending=False)['head'].astype(int)))
        heads_asc  = list(zip(sub.sort_values('importance', ascending=True)['layer'].astype(int),
                              sub.sort_values('importance', ascending=True)['head'].astype(int)))
        orderings[f'{cell_letter}_imp_desc'] = heads_desc
        orderings[f'{cell_letter}_imp_asc']  = heads_asc
    for cell_letter in ('C', 'D'):
        sub = cell_df[cell_df['cell']==cell_letter]
        heads_asc = list(zip(sub.sort_values('importance', ascending=True)['layer'].astype(int),
                             sub.sort_values('importance', ascending=True)['head'].astype(int)))
        orderings[f'{cell_letter}_imp_asc'] = heads_asc
    from scipy.stats import rankdata as _rankdata
    _low_imp = cell_df[cell_df['cell'].isin(['C', 'D'])].reset_index(drop=True)
    if len(_low_imp) > 0:
        _imp_arr = _low_imp['importance'].to_numpy()
        _pert_arr = _low_imp['perturbation'].to_numpy()
        _joint_rank = _rankdata(_imp_arr) + _rankdata(_pert_arr)
        _ordered = _low_imp.iloc[np.argsort(_joint_rank)]
        orderings['diag_rank_asc'] = list(zip(
            _ordered['layer'].astype(int), _ordered['head'].astype(int)))
    else:
        orderings['diag_rank_asc'] = []
    return orderings


def run_gamma(model, arch, tokenizer, cell_df, trials_df, clean_df, model_short):
    """Phase 4 with TRIAL_BATCH batching: each (ordering, ratio) processes B trials per trace."""
    out_path = f'{phase_dir(model_short, "30_patching")}/behavioral_gamma.csv'
    if os.path.exists(out_path):
        done = pd.read_csv(out_path)
        if 'ordering' in done.columns:
            done_keys = set(zip(done['trial_id'], done['ordering'], done['ratio'].round(4)))
            rows = list(done.to_dict('records'))
            log(f'  Phase4 resume: {len(done_keys)} (trial,ordering,ratio) triples done')
        else:
            log(f'  Phase4 WARN: existing CSV has no ordering column, starting fresh')
            done_keys = set(); rows = []
    else:
        done_keys = set(); rows = []

    total_heads = arch['total_heads']
    clean_lookup = clean_df.set_index('trial_id')
    orderings = build_orderings(cell_df)
    missing = REQUIRED_ORDERINGS - set(orderings.keys())
    if missing:
        raise RuntimeError(f'Missing orderings: {missing}')

    ordering_to_cell = {}
    for cell_letter in ('hihp', 'hilp'):
        ordering_to_cell[f'{cell_letter}_imp_desc'] = cell_letter
        ordering_to_cell[f'{cell_letter}_imp_asc']  = cell_letter
    for cell_letter in ('C', 'D'):
        ordering_to_cell[f'{cell_letter}_imp_asc'] = cell_letter
    ordering_to_cell['diag_rank_asc'] = 'diag'

    # Pre-collect corrupt_caches for ALL trials (sequential, ~80 traces)
    log(f'  Phase4 pre-collecting corrupt_caches for {len(trials_df)} trials...')
    t_pre = time.time()
    corrupt_caches = {}
    for ti, trial in trials_df.iterrows():
        tid = trial['trial_id']
        try:
            corrupt_caches[tid] = collect_attn_input_per_layer(model, arch, trial['corrupt_prompt'])
        except Exception as e:
            log(f'  Phase4 FAIL collect {tid}: {type(e).__name__}: {str(e)[:FAIL_LOG_CHARS]}')
        if (ti+1) % 10 == 0:
            log(f'    pre-collect {ti+1}/{len(trials_df)}  {_mem_status()}')
    log(f'  Phase4 pre-collect done  elapsed={time.time()-t_pre:.0f}s  cached={len(corrupt_caches)}')
    _sweep_memory()

    t0 = time.time()
    total_combos = len(ORDERINGS) * len(RATIOS)
    combo_idx = 0
    for ordering_name in ORDERINGS:
        cell_heads_sorted = orderings[ordering_name]
        cell_letter = ordering_to_cell[ordering_name]
        for ratio in RATIOS:
            combo_idx += 1
            t_combo = time.time()
            k = max(1, int(total_heads * ratio))
            k_actual = min(k, len(cell_heads_sorted))
            patch_heads = cell_heads_sorted[:k_actual]

            # Trials needing this combo
            todo_trials = []
            for ti, trial in trials_df.iterrows():
                tid = trial['trial_id']
                key = (tid, ordering_name, round(ratio, 4))
                if key in done_keys: continue
                if tid not in corrupt_caches: continue
                todo_trials.append(trial)
            if not todo_trials:
                continue

            # Batch through trials
            new_rows = []
            batch_start = 0
            while batch_start < len(todo_trials):
                def _do_batch(B):
                    nonlocal batch_start, new_rows
                    batch = todo_trials[batch_start:batch_start + B]
                    if not batch: return
                    prompts = [t['prompt'] for t in batch]
                    src_vecs = [corrupt_caches[t['trial_id']] for t in batch]
                    if not patch_heads:
                        # Empty patch (e.g., diag_rank_asc with empty C∪D) → return clean logits
                        all_logits = np.array([clean_logits_last(model, p) for p in prompts])
                    else:
                        all_logits = patched_logits_batched_trials(model, arch, prompts, src_vecs, patch_heads)
                    for tr, lg in zip(batch, all_logits):
                        tid = tr['trial_id']
                        target_id = int(clean_lookup.loc[tid, 'target_id'])
                        competitor_id = int(clean_lookup.loc[tid, 'competitor_id'])
                        clean_argmax_id = int(clean_lookup.loc[tid, 'argmax_id'])
                        clean_target_logit = float(clean_lookup.loc[tid, 'target_logit'])
                        patched_argmax = int(lg.argmax())
                        sorted_idx = np.argsort(-lg)
                        patched_rank = int(np.where(sorted_idx == target_id)[0][0]) + 1
                        new_rows.append({
                            'trial_id': tid, 'cell': cell_letter,
                            'ordering': ordering_name, 'ratio': ratio,
                            'k': k, 'k_actual': k_actual,
                            'argmax_preserved': (clean_argmax_id == patched_argmax),
                            'target_preserved_top1': (patched_argmax == target_id),
                            'patched_argmax_id': patched_argmax,
                            'patched_target_rank': patched_rank,
                            'patched_target_logit': float(lg[target_id]),
                            'patched_competitor_logit': float(lg[competitor_id]),
                            'patched_margin': float(lg[target_id] - lg[competitor_id]),
                            'clean_target_logit': clean_target_logit,
                            'delta_target_logit': float(lg[target_id]) - clean_target_logit,
                        })
                try:
                    used_b = _call_with_batch_ladder(_do_batch, 'TRIAL_BATCH_SIZE', 'TRIAL_BATCH_LADDER')
                    batch_start += used_b
                except Exception as e:
                    log(f'    Phase4 batch FAIL ord={ordering_name} r={ratio} start={batch_start}: '
                        f'{type(e).__name__}: {str(e)[:FAIL_LOG_CHARS]}')
                    batch_start += TRIAL_BATCH_SIZE  # skip

            rows.extend(new_rows)
            for r in new_rows:
                done_keys.add((r['trial_id'], r['ordering'], round(r['ratio'], 4)))
            pd.DataFrame(rows).to_csv(out_path, index=False)
            combo_dt = time.time() - t_combo
            elapsed = time.time() - t0
            log(f'  Phase4 [{combo_idx}/{total_combos}] ord={ordering_name:>15s} r={ratio:.2f} '
                f'k={k_actual} added={len(new_rows)}/{len(todo_trials)} dt={combo_dt:.0f}s '
                f'cum={elapsed:.0f}s {_mem_status()}')
            _sweep_memory()

    gamma_df = pd.DataFrame(rows)
    summary = gamma_df.groupby(['cell', 'ordering', 'ratio']).agg(
        argmax_stability=('argmax_preserved', 'mean'),
        target_top1_stability=('target_preserved_top1', 'mean'),
        mean_target_rank=('patched_target_rank', lambda x: float(x[x>=0].mean()) if (x>=0).any() else float('nan')),
        median_target_rank=('patched_target_rank', lambda x: float(x[x>=0].median()) if (x>=0).any() else float('nan')),
        mean_margin=('patched_margin', lambda x: float(np.nanmean(x))),
        mean_abs_delta=('delta_target_logit', lambda x: float(np.nanmean(np.abs(x)))),
        n_trials=('trial_id', 'count'),
    ).reset_index()
    summary.to_csv(f'{phase_dir(model_short, "30_patching")}/behavioral_gamma_summary.csv', index=False)
    log(f'  Phase4 complete  elapsed={time.time()-t0:.0f}s  {_mem_status()}')
    return gamma_df, summary


In [ ]:
# ── Cell 6: Main loop (single 70B model) ──
all_success = True
for MODEL_ID, MODEL_SHORT in MODEL_SPECS:
    log('=' * 70)
    log(f'START: {MODEL_SHORT}  ({MODEL_ID})  initial mem={_mem_status()}')
    t_model = time.time(); model = None
    try:
        model, arch = load_model(MODEL_ID)
        tokenizer = model.tokenizer

        if phase_done(MODEL_SHORT, '10_collection'):
            log(f'  [Phase 1] SKIP'); clean_df = pd.read_csv(f'{phase_dir(MODEL_SHORT, "10_collection")}/clean_logits.csv')
        else:
            _t10=time.time(); log(f'  [Phase 1] clean pass ({len(trials_df)} trials)')
            clean_df = run_clean_pass(model, arch, tokenizer, trials_df, MODEL_SHORT)
            mark_phase_done(MODEL_SHORT, '10_collection',
                {'model_id': MODEL_ID, 'model_dtype': MODEL_DTYPE, 'n_trials': len(clean_df),
                 'n_top1': int(clean_df['is_top1'].sum()),
                 'n_top5': int(clean_df['is_top5'].sum())},
                start_time=_t10)

        if phase_done(MODEL_SHORT, '20_scoring'):
            log(f'  [Phase 2] SKIP')
            imp_head = pd.read_csv(f'{phase_dir(MODEL_SHORT, "20_scoring")}/importance_per_head.csv')
            pert_head = pd.read_csv(f'{phase_dir(MODEL_SHORT, "20_scoring")}/perturbation_per_head.csv')
        else:
            _t20=time.time(); log(f'  [Phase 2] importance + perturbation per head (HEAD_BATCH={HEAD_BATCH_SIZE})')
            imp_head, pert_head = run_phase2_scoring(model, arch, tokenizer, trials_df, clean_df, MODEL_SHORT)

        cell_path = f'{phase_dir(MODEL_SHORT, "20_scoring")}/cell_classification.csv'
        if os.path.exists(cell_path):
            cell_df = pd.read_csv(cell_path)
            cell_values = set(cell_df['cell'].unique())
            if 'A' in cell_values or 'B' in cell_values:
                log(f'  [Phase 3] AUTO-MIGRATE cell_classification.csv: A→hihp, B→hilp')
                cell_df['cell'] = cell_df['cell'].replace({'A': 'hihp', 'B': 'hilp'})
                cell_df.to_csv(cell_path, index=False)
                bg_path = f'{phase_dir(MODEL_SHORT, "30_patching")}/behavioral_gamma.csv'
                if os.path.exists(bg_path):
                    bg = pd.read_csv(bg_path)
                    if 'ordering' in bg.columns:
                        invalid = bg[bg['ordering'].str.startswith(('hihp','hilp'), na=False)]
                        if len(invalid) > 0 and (invalid['k_actual'] == 0).any():
                            log(f'  [Phase 4] invalid rows detected, dropping {len(invalid)} rows')
                            bg_clean = bg[~bg['ordering'].str.startswith(('hihp','hilp'), na=False)]
                            bg_clean.to_csv(bg_path, index=False)
            else:
                log(f'  [Phase 3] SKIP (cell_classification.csv present)')
        else:
            log(f'  [Phase 3] cell classification (median split)')
            cell_df = classify_cells(imp_head, pert_head, MODEL_SHORT)

        if not phase_done(MODEL_SHORT, '20_scoring'):
            mark_phase_done(MODEL_SHORT, '20_scoring',
                {'model_id': MODEL_ID, 'model_dtype': MODEL_DTYPE,
                 'n_heads': int(arch['total_heads']),
                 'head_batch_size_used': HEAD_BATCH_SIZE},
                start_time=locals().get('_t20'))

        if phase4_all_orderings_complete(MODEL_SHORT):
            log(f'  [Phase 4] SKIP (all {len(REQUIRED_ORDERINGS)} orderings complete)')
        else:
            _t40=time.time(); log(f'  [Phase 4] behavioral γ  cells={CELLS}  orderings={ORDERINGS}  ratios={RATIOS}  TRIAL_BATCH={TRIAL_BATCH_SIZE}')
            run_gamma(model, arch, tokenizer, cell_df, trials_df, clean_df, MODEL_SHORT)
            update_phase4_config(MODEL_SHORT,
                {'model_id': MODEL_ID, 'model_dtype': MODEL_DTYPE,
                 'ratios': RATIOS, 'cells': CELLS, 'orderings': ORDERINGS,
                 'trial_batch_size_used': TRIAL_BATCH_SIZE,
                 'phase4_elapsed_sec': round(time.time()-_t40, 2)})

        log(f'  DONE: {MODEL_SHORT}  ({(time.time()-t_model)/60:.1f} min)  final_mem={_mem_status()}')
    except Exception as e:
        log(f'  FAILED: {MODEL_SHORT}  {type(e).__name__}: {str(e)[:500]}')
        traceback.print_exc(); all_success = False
    finally:
        if model is not None:
            unload_model(model)

log('=' * 70)
log(f'ALL MODELS PROCESSED.  overall_success={all_success}  final_mem={_mem_status()}')


In [ ]:
# ── Cell 7: Audio beep + runtime.unassign() only on success ──
try:
    from IPython.display import Audio, display
    sr = 44100; _t = np.linspace(0, 1, sr)
    display(Audio(0.5 * np.sin(2 * np.pi * 440 * _t), rate=sr, autoplay=True))
except Exception as e:
    log(f'beep failed: {e}')

if all_success:
    log('All models complete. Unassigning Colab runtime.')
    runtime.unassign()
else:
    log('Some models failed; keeping runtime alive for inspection.')
